In [1]:
# ============================================================
# 1. 2D Elasticity MMS Benchmark：定义物理参数与制造解
#
# 目标：
# - 构造一个有精确 Ground Truth 的二维线弹性问题
# - Residual PINN 使用强形式 PDE residual（二阶 AD）
# - Energy PINN 使用总势能（一阶 AD + 积分）
#
# 区域：
#     (x, y) ∈ [0, 1] × [0, 1]
#
# 制造位移解：
#     u_x* = sin(pi x) sin(pi y)
#     u_y* = sin(pi x) sin(pi y)
#
# 该解在四条边界上都严格为 0，
# 因此 Dirichlet 边界非常干净，不存在刚体自由度问题。
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn


# ==============================
# 环境与设备
# ==============================

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))


# ==============================
# 几何参数
# ==============================

Lx = 1.0
Ly = 1.0


# ==============================
# 材料参数：Plane Strain
# ==============================

E = 1.0
nu = 0.3

lam = E * nu / (
    (1.0 + nu) * (1.0 - 2.0 * nu)
)

mu = E / (
    2.0 * (1.0 + nu)
)

print("lambda =", lam)
print("mu     =", mu)


# ==============================
# 制造位移解析解
# ==============================

def exact_displacement(X):
    """
    MMS 精确位移解

    输入:
        X.shape = [N, 2]

    返回:
        u_x_exact, u_y_exact
    """

    x = X[:, 0:1]
    y = X[:, 1:2]

    u_x = (
        torch.sin(torch.pi * x)
        *
        torch.sin(torch.pi * y)
    )

    u_y = (
        torch.sin(torch.pi * x)
        *
        torch.sin(torch.pi * y)
    )

    return u_x, u_y

PyTorch: 2.10.0+cu128
Device : cuda
GPU    : NVIDIA GeForce RTX 5060
lambda = 0.5769230769230769
mu     = 0.3846153846153846
